# Lesson 8.1 — Designing HA & Fault-Tolerant Architectures

**Level: Advanced** | **Exam:** SAP-C02 Domain 2, Tasks 2.2 & 2.4

## High Availability vs Fault Tolerance

| | High Availability | Fault Tolerance |
|--|-----------------|----------------|
| **Goal** | Minimize downtime | Zero downtime |
| **Approach** | Fast failover | Continuous operation despite failure |
| **Example** | RDS Multi-AZ (1-2 min failover) | DynamoDB (no failover needed) |
| **Cost** | Lower | Higher |

## Designing for HA — Key Patterns

### Eliminate Single Points of Failure
```
SPOF              → HA Solution
Single EC2        → ASG with ALB across 2+ AZs
Single RDS        → RDS Multi-AZ
Single AZ subnet  → Multi-AZ subnets
Single NAT GW     → NAT GW per AZ (or NAT instance HA)
Single DNS        → Route 53 with health checks
```

### Health Check Cascade
```
Application Layer: Lambda → CloudWatch → Alarm → SSM Automation → remediation
Compute Layer:     EC2 → ELB health check → ASG replacement
Database Layer:    RDS Multi-AZ → automatic failover
DNS Layer:         Route 53 health check → failover record
```

### Availability SLA Math
```
99.9% = 8.7 hours downtime/year
99.99% = 52 minutes downtime/year
99.999% (five nines) = 5 minutes downtime/year

2 components in series (both must work):
  99.9% x 99.9% = 99.8% (slightly worse!)

2 components in parallel (either works):
  1 - (0.001 x 0.001) = 99.9999% (much better!)
```
=> Parallel architectures dramatically improve availability.

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Analyze current architecture for HA patterns
import boto3, json

ec2 = boto3.client('ec2', region_name='us-east-1')
elbv2 = boto3.client('elbv2', region_name='us-east-1')
rds = boto3.client('rds', region_name='us-east-1')

print("=== HA Architecture Audit ===")
print()

# Check ASGs
autoscaling = boto3.client('autoscaling', region_name='us-east-1')
asgs = autoscaling.describe_auto_scaling_groups()['AutoScalingGroups']
print(f"Auto Scaling Groups: {len(asgs)}")
for asg in asgs:
    ha = len(asg['AvailabilityZones']) > 1
    health_type = asg['HealthCheckType']
    print(f"  {asg['AutoScalingGroupName']}: {len(asg['AvailabilityZones'])} AZs, health={health_type} {'[HA OK]' if ha else '[SPOF RISK]'}")

# Check RDS
dbs = rds.describe_db_instances()['DBInstances']
print(f"\nRDS Instances: {len(dbs)}")
for db in dbs:
    multi_az = "Multi-AZ" if db['MultiAZ'] else "SINGLE-AZ [RISK]"
    print(f"  {db['DBInstanceIdentifier']}: {multi_az}")

# Check ELBs
lbs = elbv2.describe_load_balancers()['LoadBalancers']
print(f"\nLoad Balancers: {len(lbs)}")
for lb in lbs:
    az_count = len(lb['AvailabilityZones'])
    print(f"  {lb['LoadBalancerName']}: {az_count} AZs")

## Exercise

In [ ]:
# TODO: Calculate the theoretical availability of these architectures
# Use the formula: parallel = 1 - (product of individual unavailabilities)
#                  series = product of individual availabilities

import math

def availability(uptime_percent):
    return uptime_percent / 100.0

def series(*components):
    """Combined availability when ALL components must work."""
    result = 1.0
    for c in components:
        result *= availability(c)
    return result * 100

def parallel(*components):
    """Combined availability when ANY ONE component can handle the load."""
    result = 1.0
    for c in components:
        result *= (1 - availability(c))
    return (1 - result) * 100

# TODO: Calculate availability for these scenarios:
# 1. Single EC2 (99.9%) + Single RDS (99.95%) in series
# 2. Same EC2 in ASG: 3x EC2 in parallel (any one can serve) + RDS Multi-AZ (99.99%)
# 3. Multi-region active/active: 2 regions, each with 99.99% availability, in parallel

# YOUR CODE HERE
print("Architecture Availability Analysis:")

## Summary
- **Eliminate SPOFs**: ASG across AZs, RDS Multi-AZ, multi-AZ NAT Gateway, Route 53 health checks
- **Parallel > Series**: Adding redundant parallel components dramatically improves availability
- The 4 DR patterns apply here: Backup/Restore, Pilot Light, Warm Standby, Active/Active
- **Design for failure**: Assume components will fail; design so failures are invisible to users
- Use **chaos engineering** to verify HA: intentionally kill instances to test failover

---
**Next:** [Lesson 8.2 — Performance Architecture: Caching & Purpose-Built Databases](lesson-8.2-performance-databases.ipynb)